In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from collections import Counter

INPUT = "clean_dataset.csv"

# Class list is read from label_encoder.json (stage 3's output) rather than
# hardcoded, so this never silently disagrees with whatever KEEP_SABOTAGE
# was set to in 02_clean_and_merge_sources.py.
with open("label_encoder.json") as _f:
    _label_map = json.load(_f)
KEEP = sorted(_label_map, key=_label_map.get)

matplotlib.rcParams.update({
    "font.family": "serif", "font.size": 11, "axes.titlesize": 13,
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
})
C_ESP, C_FIN, C_GREY = "#2166AC", "#D6604D", "#737373"
os.makedirs("figures", exist_ok=True)

df = pd.read_csv(INPUT)
df = df[df["motive"].isin(KEEP)].reset_index(drop=True)
df["techs"] = df["technique_id_seq"].apply(lambda s: s.split(" -> "))
df["n"] = df["techs"].apply(len)

esp = df[df["motive"] == "Espionage"]
fin = df[df["motive"] == "Financial"]

# ── 1. Class + source composition ─────────────────────────────────────────────
print("=== Class composition ===")
print(df["motive"].value_counts().to_string())
print(f"Total actors: {len(df)}")
if "source_type" in df.columns:
    print("\n=== Source-type composition ===")
    print(df["source_type"].value_counts().to_string())

    # Finer breakdown for the manually-collected (bounded_incident) rows only,
    # derived from the source URL domain, matching the thesis's existing
    # tab:source-subcategory table (MITRE Campaign objects / DFIR reports /
    # vendor write-ups). Tropchaud rows are excluded here since their
    # "source" value is just the literal string "Tropchaud", not a URL.
    def _domain_category(url):
        url = str(url)
        if "attack.mitre.org" in url:
            return "MITRE ATT&CK Campaign objects"
        if "thedfirreport.com" in url:
            return "DFIR incident reports"
        return "Vendor write-ups / advisories"

    bounded = df[df["source_type"] == "bounded_incident"]
    print("\n=== Manually-collected entries by underlying source type ===")
    print(bounded["source"].apply(_domain_category).value_counts().to_string())

# ── 2. Sequence-length statistics by motive ───────────────────────────────────
print("\n=== Sequence length by motive ===")
for name, g in [("Espionage", esp), ("Financial", fin), ("All", df)]:
    s = g["n"]
    print(f"{name:<10} mean {s.mean():5.1f}  median {s.median():4.0f}  "
          f"min {s.min():2d}  max {s.max():3d}  std {s.std():5.1f}")

# ── 3. Sequence-length histogram by motive ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bins = np.arange(0, df["n"].max() + 5, 5)
ax.hist(esp["n"], bins=bins, alpha=0.6, color=C_ESP, label="Espionage",
        edgecolor="white", linewidth=0.5)
ax.hist(fin["n"], bins=bins, alpha=0.6, color=C_FIN, label="Financial",
        edgecolor="white", linewidth=0.5)
ax.axvline(esp["n"].median(), color=C_ESP, ls="--", lw=1.5,
           label=f"Espionage median ({esp['n'].median():.0f})")
ax.axvline(fin["n"].median(), color=C_FIN, ls="--", lw=1.5,
           label=f"Financial median ({fin['n'].median():.0f})")
ax.set_xlabel("Number of techniques per actor")
ax.set_ylabel("Number of actors")
ax.set_title("Sequence-length distribution by motive")
ax.legend()
fig.savefig("figures/seq_length_by_motive.png")
fig.savefig("figures/seq_length_by_motive.pdf")
plt.close(fig)
print("\n[✓] Saved: figures/seq_length_by_motive.png + .pdf")

# ── 4. Most discriminative techniques (log-odds with smoothing) ───────────────
# For each technique, compare P(appears | Financial) vs P(appears | Espionage)
# at the actor level. Log-odds ratio, Laplace-smoothed. Restrict to techniques
# appearing in at least MIN_ACTORS actors to avoid noise from rare IDs.
MIN_ACTORS = 6
esp_counts = Counter(t for techs in esp["techs"] for t in set(techs))
fin_counts = Counter(t for techs in fin["techs"] for t in set(techs))
n_esp, n_fin = len(esp), len(fin)

rows = []
all_techs = set(esp_counts) | set(fin_counts)
for t in all_techs:
    total = esp_counts.get(t, 0) + fin_counts.get(t, 0)
    if total < MIN_ACTORS:
        continue
    p_fin = (fin_counts.get(t, 0) + 1) / (n_fin + 2)
    p_esp = (esp_counts.get(t, 0) + 1) / (n_esp + 2)
    logodds = np.log(p_fin / (1 - p_fin)) - np.log(p_esp / (1 - p_esp))
    rows.append((t, logodds, fin_counts.get(t, 0), esp_counts.get(t, 0)))

tbl = pd.DataFrame(rows, columns=["technique", "logodds", "fin_actors", "esp_actors"])
tbl = tbl.sort_values("logodds")

top_fin = tbl.tail(8).iloc[::-1]   # most Financial-indicative
top_esp = tbl.head(8)              # most Espionage-indicative

print(f"\n=== Technique vocabulary: {len(all_techs)} unique parent techniques ===")
print("\nMost FINANCIAL-indicative techniques (log-odds > 0):")
print(top_fin.to_string(index=False))
print("\nMost ESPIONAGE-indicative techniques (log-odds < 0):")
print(top_esp.to_string(index=False))

# ── 5. Discriminative-technique bar chart ─────────────────────────────────────
plot_df = pd.concat([top_esp, top_fin]).sort_values("logodds")
colors = [C_ESP if v < 0 else C_FIN for v in plot_df["logodds"]]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(plot_df["technique"], plot_df["logodds"], color=colors, alpha=0.85)
ax.axvline(0, color=C_GREY, lw=1)
ax.set_xlabel("Log-odds  (← Espionage-indicative    Financial-indicative →)")
ax.set_ylabel("MITRE ATT&CK technique ID")
ax.set_title("Most class-indicative techniques (actor-level log-odds)")
fig.savefig("figures/discriminative_techniques.png")
fig.savefig("figures/discriminative_techniques.pdf")
plt.close(fig)
print("\n[✓] Saved: figures/discriminative_techniques.png + .pdf")
